In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast
import os

# ----------------------------------------------
# Configuration
# ----------------------------------------------
base_path_template = '../data/agnews/{steps}_steps/evolution/'
seeds = [42, 123, 999, 2024, 7, 2026, 22]
run_ids = [1, 2, 3, 4, 5, 6, 7]
step_budgets = [100, 300, 400]
OUTPUT_DIR = '../data/agnews/combined_plots/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------
# Helpers (unchanged)
# ----------------------------------------------
def parse_genotype(gen_str):
    try:
        return ast.literal_eval(gen_str)
    except:
        return []

def load_and_aggregate(steps):
    all_runs = []
    prefix = 'results_run_'
    suffix = f'_{steps}_steps.csv'
    base_path = base_path_template.format(steps=steps)
    for rid, seed in zip(run_ids, seeds):
        fname = f'{base_path}{prefix}{rid}_seed_{seed}{suffix}'
        df = pd.read_csv(fname)
        df['genotype_list'] = df['genotype'].apply(parse_genotype)
        df['num_0'] = df['genotype_list'].apply(lambda g: g.count(0))
        df['num_1'] = df['genotype_list'].apply(lambda g: g.count(1))
        df['depth'] = df['genotype_list'].apply(len)
        all_runs.append(df)
    return all_runs

def aggregate_per_run(df):
    gen = df.groupby('generation')
    agg = gen.agg(
        fitness_mean = ('fitness', 'mean'),
        fitness_max  = ('fitness', 'max'),
        depth_mean   = ('depth', 'mean'),
        depth_max    = ('depth', 'max'),
        num_0_mean   = ('num_0', 'mean'),
        num_1_mean   = ('num_1', 'mean')
    ).reset_index()
    agg['pct_0'] = agg['num_0_mean'] / (agg['num_0_mean'] + agg['num_1_mean']) * 100
    agg['pct_1'] = 100 - agg['pct_0']
    return agg

def across_runs(agg_list, columns):
    merged = pd.concat(agg_list)
    grouped = merged.groupby('generation')
    result = pd.DataFrame()
    for col in columns:
        result[col + '_mean'] = grouped[col].mean()
        result[col + '_std']  = grouped[col].std()
    return result.reset_index()

def plot_with_band(ax, x, y_mean, y_std, label, color, alpha=0.3):
    ax.plot(x, y_mean, label=label, color=color)
    ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=alpha, color=color)

# ----------------------------------------------
# Collect aggregated data
# ----------------------------------------------
agg_dict = {}
for steps in step_budgets:
    runs = load_and_aggregate(steps)
    agg_per = [aggregate_per_run(df) for df in runs]
    pop_cols = ['fitness_mean', 'fitness_max', 'depth_mean', 'depth_max', 'pct_0', 'pct_1']
    agg_dict[steps] = across_runs(agg_per, pop_cols)

# ----------------------------------------------
# Helper to create 2x2 grid with bottom-center subplot
# ----------------------------------------------
def make_figure(title, ylabel, plot_func):
    fig = plt.figure(figsize=(16, 12))
    # Two subplots on top, one centered subplot on the bottom (spanning both columns)
    ax1 = plt.subplot2grid((2, 2), (0, 0))
    ax2 = plt.subplot2grid((2, 2), (0, 1))
    ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)
    # The bottom‑right cell is automatically empty – no need to create or delete anything

    axes = [ax1, ax2, ax3]
    for ax, steps in zip(axes, step_budgets):
        data = agg_dict[steps]
        plot_func(ax, data, steps)

    fig.suptitle(title, fontsize=16, y=0.98)
    fig.text(0.5, 0.04, 'Generation', ha='center', fontsize=14)
    fig.text(0.04, 0.5, ylabel, va='center', rotation='vertical', fontsize=14)
    plt.tight_layout(rect=[0.05, 0.06, 1, 0.93])
    return fig

# ----------------------------------------------
# Figure 1: Fitness evolution
# ----------------------------------------------
def plot_fitness(ax, data, steps):
    plot_with_band(ax, data['generation'], data['fitness_mean_mean'], data['fitness_mean_std'],
                   'Mean fitness', 'blue')
    plot_with_band(ax, data['generation'], data['fitness_max_mean'], data['fitness_max_std'],
                   'Best fitness', 'red')
    ax.set_title(f'{steps} steps')
    ax.legend()

fig1 = make_figure('Fitness Evolution', 'Weighted F1', plot_fitness)
fig1.savefig(OUTPUT_DIR + 'fitness_evolution_combined.png', dpi=150)
plt.close(fig1)

# ----------------------------------------------
# Figure 2: Average depth evolution
# ----------------------------------------------
def plot_depth(ax, data, steps):
    plot_with_band(ax, data['generation'], data['depth_mean_mean'], data['depth_mean_std'],
                   'Mean depth', 'purple')
    plot_with_band(ax, data['generation'], data['depth_max_mean'], data['depth_max_std'],
                   'Max depth', 'brown')
    ax.set_title(f'{steps} steps')
    ax.legend()

fig2 = make_figure('Average Depth Evolution', 'Layers', plot_depth)
fig2.savefig(OUTPUT_DIR + 'depth_evolution_combined.png', dpi=150)
plt.close(fig2)

# ----------------------------------------------
# Figure 3: Layer type proportion
# ----------------------------------------------
def plot_proportion(ax, data, steps):
    ax.fill_between(data['generation'], 0, data['pct_0_mean'],
                    label='Mamba (0)', color='skyblue', alpha=0.6)
    ax.fill_between(data['generation'], data['pct_0_mean'], 100,
                    label='Attention (1)', color='salmon', alpha=0.6)
    ax.set_title(f'{steps} steps')
    ax.legend()

fig3 = make_figure('Layer Type Proportion', 'Percentage (%)', plot_proportion)
fig3.savefig(OUTPUT_DIR + 'proportion_combined.png', dpi=150)
plt.close(fig3)

print(f"Combined plots saved in {OUTPUT_DIR}")

Combined plots saved in ../data/agnews/combined_plots/


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast
import os

# ----------------------------------------------
# Configuration – PROPOR (400 steps)
# ----------------------------------------------
base_path = '../data/propor/400_steps/evolution/'
seeds = [42, 123, 999, 2024, 7, 2026, 22]
run_ids = [1, 2, 3, 4, 5, 6, 7]
prefix = 'results_run_'
suffix = '_propor.csv'             # adapt to the actual file suffix used for PROPOR
OUTPUT_DIR = '../data/propor/400_steps/plots/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------
# Helpers
# ----------------------------------------------
def parse_genotype(gen_str):
    try:
        return ast.literal_eval(gen_str)
    except:
        return []

def load_all_runs():
    all_runs = []
    for rid, seed in zip(run_ids, seeds):
        fname = f'{base_path}{prefix}{rid}_seed_{seed}{suffix}'
        df = pd.read_csv(fname)
        df['genotype_list'] = df['genotype'].apply(parse_genotype)
        df['num_0'] = df['genotype_list'].apply(lambda g: g.count(0))
        df['num_1'] = df['genotype_list'].apply(lambda g: g.count(1))
        df['depth'] = df['genotype_list'].apply(len)
        all_runs.append(df)
    return all_runs

def aggregate_per_run(df):
    gen = df.groupby('generation')
    agg = gen.agg(
        fitness_mean = ('fitness', 'mean'),
        fitness_max  = ('fitness', 'max'),
        depth_mean   = ('depth', 'mean'),
        depth_max    = ('depth', 'max'),
        num_0_mean   = ('num_0', 'mean'),
        num_1_mean   = ('num_1', 'mean')
    ).reset_index()
    agg['pct_0'] = agg['num_0_mean'] / (agg['num_0_mean'] + agg['num_1_mean']) * 100
    agg['pct_1'] = 100 - agg['pct_0']
    return agg

def across_runs(agg_list, columns):
    merged = pd.concat(agg_list)
    grouped = merged.groupby('generation')
    result = pd.DataFrame()
    for col in columns:
        result[col + '_mean'] = grouped[col].mean()
        result[col + '_std']  = grouped[col].std()
    return result.reset_index()

def plot_with_band(ax, x, y_mean, y_std, label, color, alpha=0.3):
    ax.plot(x, y_mean, label=label, color=color)
    ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=alpha, color=color)

# ----------------------------------------------
# Load and aggregate PROPOR data
# ----------------------------------------------
print("Loading PROPOR runs...")
all_runs = load_all_runs()
agg_per = [aggregate_per_run(df) for df in all_runs]
pop_cols = ['fitness_mean', 'fitness_max', 'depth_mean', 'depth_max', 'pct_0', 'pct_1']
data = across_runs(agg_per, pop_cols)
gen = data['generation']

# ----------------------------------------------
# Figure 1: Fitness evolution (PROPOR, 400 steps)
# ----------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
plot_with_band(ax, gen, data['fitness_mean_mean'], data['fitness_mean_std'],
               'Mean fitness', 'blue')
plot_with_band(ax, gen, data['fitness_max_mean'], data['fitness_max_std'],
               'Best fitness', 'red')
ax.set_title('Fitness Evolution – PROPOR (400 steps)')
ax.set_xlabel('Generation')
ax.set_ylabel('Weighted F1')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fitness_evolution_propor.png', dpi=150)
plt.close()

# ----------------------------------------------
# Figure 2: Average depth evolution
# ----------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
plot_with_band(ax, gen, data['depth_mean_mean'], data['depth_mean_std'],
               'Mean depth', 'purple')
plot_with_band(ax, gen, data['depth_max_mean'], data['depth_max_std'],
               'Max depth', 'brown')
ax.set_title('Average Depth Evolution – PROPOR (400 steps)')
ax.set_xlabel('Generation')
ax.set_ylabel('Layers')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'depth_evolution_propor.png', dpi=150)
plt.close()

# ----------------------------------------------
# Figure 3: Layer type proportion
# ----------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(gen, 0, data['pct_0_mean'], label='Mamba (0)',
                color='skyblue', alpha=0.6)
ax.fill_between(gen, data['pct_0_mean'], 100, label='Attention (1)',
                color='salmon', alpha=0.6)
ax.set_title('Layer Type Proportion – PROPOR (400 steps)')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage (%)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'proportion_propor.png', dpi=150)
plt.close()

print(f"PROPOR plots saved in {OUTPUT_DIR}")

Loading PROPOR runs...
PROPOR plots saved in ../data/propor/400_steps/plots/
